# 🎯 Student Exam Score Prediction with PKBoost

**Competition:** Playground Series - Season 6, Episode 1  
**Goal:** Predict students' test scores (RMSE evaluation)  
**Engine:** PKBoost - Shannon-guided gradient boosting with zero-copy performance

---

## 1. Setup & Installation

In [ ]:
# Install PKBoost (if not already installed)
!pip install pkboost -q

import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# PKBoost import
from pkboost_sklearn import PKBoostRegressor

print("✅ All imports successful!")

## 2. Load Data

In [ ]:
# Load competition data
train = pd.read_csv('/kaggle/input/playground-series-s6e1/train.csv')
test = pd.read_csv('/kaggle/input/playground-series-s6e1/test.csv')
sample_sub = pd.read_csv('/kaggle/input/playground-series-s6e1/sample_submission.csv')

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")
print(f"Target: exam_score")
train.head()

## 3. Exploratory Data Analysis

In [ ]:
# Basic info
print("=" * 50)
print("DATASET INFO")
print("=" * 50)
print(f"\nTrain samples: {len(train):,}")
print(f"Test samples: {len(test):,}")
print(f"Features: {train.shape[1] - 2}")  # minus id and target

# Target distribution
print(f"\nTarget Statistics:")
print(train['exam_score'].describe())

# Missing values
print(f"\nMissing values in train: {train.isnull().sum().sum()}")
print(f"Missing values in test: {test.isnull().sum().sum()}")

In [ ]:
# Column types
print("\nColumn Types:")
for col in train.columns:
    if col not in ['id', 'exam_score']:
        dtype = train[col].dtype
        nunique = train[col].nunique()
        print(f"  {col}: {dtype} ({nunique} unique)")

## 4. Feature Engineering

In [ ]:
def preprocess_data(train_df, test_df):
    """Preprocess train and test data."""
    
    # Combine for consistent encoding
    train_df = train_df.copy()
    test_df = test_df.copy()
    
    # Store IDs and target
    train_ids = train_df['id'].values
    test_ids = test_df['id'].values
    target = train_df['exam_score'].values
    
    # Drop id and target
    train_df = train_df.drop(['id', 'exam_score'], axis=1)
    test_df = test_df.drop(['id'], axis=1)
    
    # Identify categorical columns
    cat_cols = train_df.select_dtypes(include=['object']).columns.tolist()
    num_cols = train_df.select_dtypes(include=['int64', 'float64']).columns.tolist()
    
    print(f"Categorical columns: {cat_cols}")
    print(f"Numerical columns: {num_cols}")
    
    # Label encode categorical columns
    label_encoders = {}
    for col in cat_cols:
        le = LabelEncoder()
        # Fit on combined data to handle unseen categories
        combined = pd.concat([train_df[col], test_df[col]], axis=0).astype(str)
        le.fit(combined)
        train_df[col] = le.transform(train_df[col].astype(str))
        test_df[col] = le.transform(test_df[col].astype(str))
        label_encoders[col] = le
    
    # Feature engineering - interaction features
    if 'Hours_Studied' in train_df.columns and 'Attendance' in train_df.columns:
        train_df['Study_Attendance_Ratio'] = train_df['Hours_Studied'] / (train_df['Attendance'] + 1)
        test_df['Study_Attendance_Ratio'] = test_df['Hours_Studied'] / (test_df['Attendance'] + 1)
    
    if 'Hours_Studied' in train_df.columns and 'Sleep_Hours' in train_df.columns:
        train_df['Study_Sleep_Balance'] = train_df['Hours_Studied'] / (train_df['Sleep_Hours'] + 1)
        test_df['Study_Sleep_Balance'] = test_df['Hours_Studied'] / (test_df['Sleep_Hours'] + 1)
    
    if 'Previous_Scores' in train_df.columns:
        train_df['Score_Improvement_Potential'] = 100 - train_df['Previous_Scores']
        test_df['Score_Improvement_Potential'] = 100 - test_df['Previous_Scores']
    
    # Convert to numpy arrays (contiguous float64 for PKBoost zero-copy)
    X_train = np.ascontiguousarray(train_df.values, dtype=np.float64)
    X_test = np.ascontiguousarray(test_df.values, dtype=np.float64)
    y_train = np.ascontiguousarray(target, dtype=np.float64)
    
    print(f"\nFinal X_train shape: {X_train.shape}")
    print(f"Final X_test shape: {X_test.shape}")
    
    return X_train, y_train, X_test, test_ids, train_df.columns.tolist()

X_train, y_train, X_test, test_ids, feature_names = preprocess_data(train, test)

## 5. PKBoost Model Training

In [ ]:
# Cross-validation with PKBoost
n_folds = 5
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

oof_preds = np.zeros(len(X_train))
test_preds = np.zeros(len(X_test))
rmse_scores = []

print("=" * 50)
print("PKBoost Cross-Validation Training")
print("=" * 50)

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train)):
    print(f"\n📊 Fold {fold + 1}/{n_folds}")
    
    # Split data
    X_tr, X_val = X_train[train_idx], X_train[val_idx]
    y_tr, y_val = y_train[train_idx], y_train[val_idx]
    
    # Create PKBoost Regressor with optimized hyperparameters
    model = PKBoostRegressor()
    
    # Train
    model.fit(X_tr, y_tr, eval_set=(X_val, y_val), verbose=True)
    
    # Predict
    val_pred = model.predict(X_val)
    test_pred = model.predict(X_test)
    
    # Store predictions
    oof_preds[val_idx] = val_pred
    test_preds += test_pred / n_folds
    
    # Calculate RMSE
    fold_rmse = np.sqrt(mean_squared_error(y_val, val_pred))
    rmse_scores.append(fold_rmse)
    print(f"   RMSE: {fold_rmse:.5f}")

print("\n" + "=" * 50)
print(f"📈 CV RMSE: {np.mean(rmse_scores):.5f} ± {np.std(rmse_scores):.5f}")
print("=" * 50)

## 6. Ensemble with Multiple Seeds (Optional)

In [ ]:
# Multi-seed ensemble for more robust predictions
SEEDS = [42, 123, 456, 789, 2024]
ensemble_preds = np.zeros(len(X_test))

print("🔄 Running multi-seed ensemble...")

for seed in SEEDS:
    kf = KFold(n_splits=5, shuffle=True, random_state=seed)
    seed_preds = np.zeros(len(X_test))
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X_train)):
        X_tr, X_val = X_train[train_idx], X_train[val_idx]
        y_tr, y_val = y_train[train_idx], y_train[val_idx]
        
        model = PKBoostRegressor()
        model.fit(X_tr, y_tr, eval_set=(X_val, y_val), verbose=False)
        
        seed_preds += model.predict(X_test) / 5
    
    ensemble_preds += seed_preds / len(SEEDS)
    print(f"  Seed {seed} complete")

print("\n✅ Ensemble complete!")

## 7. Create Submission

In [ ]:
# Create submission using ensemble predictions
submission = pd.DataFrame({
    'id': test_ids,
    'exam_score': ensemble_preds
})

# Clip predictions to valid range (0-100 for exam scores)
submission['exam_score'] = submission['exam_score'].clip(0, 100)

# Save submission
submission.to_csv('submission.csv', index=False)

print("📁 Submission saved!")
print(f"\nSubmission shape: {submission.shape}")
print(f"\nPrediction statistics:")
print(submission['exam_score'].describe())

submission.head(10)

## 8. Post-Processing (Optional Refinement)

In [ ]:
# Blend with simple average if desired
# This cell is optional - only use if you have other model predictions to blend

# # Example: blend PKBoost with another model
# from sklearn.ensemble import GradientBoostingRegressor
# 
# gb = GradientBoostingRegressor(n_estimators=200, random_state=42)
# gb.fit(X_train, y_train)
# gb_preds = gb.predict(X_test)
# 
# # Blend: 70% PKBoost, 30% GB
# final_preds = 0.7 * ensemble_preds + 0.3 * gb_preds

print("💡 Tip: Consider blending with other models for potentially better results!")

---

## Summary

This notebook uses **PKBoost** - a Shannon entropy-guided gradient boosting library with:
- 🚀 Zero-copy data transfer from NumPy
- 📊 Optimized for regression tasks
- ⚡ Rust-powered performance

**Key techniques:**
1. Label encoding for categorical features
2. Feature engineering (interaction features)
3. 5-fold cross-validation
4. Multi-seed ensemble for robust predictions

Good luck! 🍀